# Vehicle Insurance Policy Lapse Prediction

## Advanced Modeling & Feature Engineering

This notebook focuses on improving the baseline machine learning pipeline using domain-driven feature engineering, preprocessing refinement, and model optimization for vehicle insurance policy lapse prediction.

Based on the baseline model comparison, Logistic Regression was selected as the primary model due to its:
- strong recall performance on the imbalanced dataset
- interpretability and business explainability
- stable generalization behavior
- suitability for structured tabular insurance data

Feature engineering focused on creating stable, interpretable, and business relevant features while reducing sparsity and multicollinearity to improve model performance and reliability.

In [1]:
# Standard Library
from pathlib import Path
import joblib
import os

# Data Manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

np.seterr(divide="ignore", invalid="ignore")

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

In [3]:
# Machine Learning - Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

# Experiment Tracking
import mlflow

# Machine Learning - Model Selection
from sklearn.model_selection import train_test_split

# Machine Learning - Models
from sklearn.linear_model import LogisticRegression

In [4]:
# Machine Learning - Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Display Settings
pd.set_option("display.max_columns", None)

# Visualization Settings
sns.set_style("whitegrid")

### Load Data

Load the dataset and create a working copy for preprocessing while preserving the original data.

In [5]:
base_dir = Path().resolve().parent

data_path = base_dir / "data" / "eudirectlapse.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {data_path}")

df = pd.read_csv(data_path)

df.head()

,lapse,polholder_age,polholder_BMCevol,polholder_diffdriver,polholder_gender,polholder_job,policy_age,policy_caruse,policy_nbcontract,prem_final,prem_freqperyear,prem_last,prem_market,prem_pure,vehicl_age,vehicl_agepurchase,vehicl_garage,vehicl_powerkw,vehicl_region
0,0,38,stable,only partner,Male,normal,1,private or freelance work,1,232.46,4 per year,232.47,221.56,243.59,9,8,private garage,225 kW,Reg7
1,1,35,stable,same,Male,normal,1,private or freelance work,1,208.53,4 per year,208.54,247.56,208.54,15,7,private garage,100 kW,Reg4
2,1,29,stable,same,Male,normal,0,private or freelance work,1,277.34,1 per year,277.35,293.32,277.35,14,6,underground garage,100 kW,Reg7
3,0,33,down,same,Female,medical,2,private or freelance work,1,239.51,4 per year,244.40,310.91,219.95,17,10,street,75 kW,Reg5
4,0,50,stable,same,Male,normal,8,unknown,1,554.54,4 per year,554.55,365.46,519.50,16,8,street,75 kW,Reg14


### Train-Test Split

The dataset is split into training and testing sets before preprocessing to avoid data leakage.

Stratified sampling is applied to maintain the original class distribution of the target variable across both sets.

In [6]:
X = df.drop(columns="lapse")
y = df["lapse"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: (18448, 18)
Test set shape: (4612, 18)


## Feature Engineering

#### Driver Risk Grouping

In [7]:
df["polholder_diffdriver"].value_counts(dropna=False)

polholder_diffdriver
same                11155
only partner         8128
young drivers        1955
all drivers > 24     1728
learner 17             42
commercial             40
unknown                12
Name: count, dtype: int64

The original `polholder_diffdriver` feature showed clear variation in lapse behavior across driver profiles, with younger and non-standard driver groups exhibiting higher lapse rates.

Several categories also contained very few observations, increasing the risk of unstable coefficients in Logistic Regression.

To improve model stability, reduce sparsity, and preserve meaningful behavioral signal, the feature was grouped into broader risk-based categories:

- `standard`
  - same
  - only partner
  - all drivers > 24

- `high_risk`
  - young drivers
  - learner 17
  - unknown
  - commercial

This transformation created a more interpretable and statistically stable representation of driver-related risk for lapse prediction.

In [8]:
high_risk = ["young drivers", "learner 17", "unknown", "commercial"]

X_train["driver_risk_group"] = np.where(
    X_train["polholder_diffdriver"].isin(high_risk),
    "high_risk",
    "standard"
)

X_test["driver_risk_group"] = np.where(
    X_test["polholder_diffdriver"].isin(high_risk),
    "high_risk",
    "standard"
)

#### Vehicle Usage Grouping


In [9]:
df["policy_caruse"].value_counts(dropna=False)

policy_caruse
private or freelance work    19567
unknown                       3483
commercial                      10
Name: count, dtype: int64


The original `policy_caruse` feature was highly imbalanced, with `commercial` containing very few observations and `unknown` representing unclear vehicle usage behavior.

To reduce sparsity and improve model stability, the feature was grouped into broader categories:
- `private`
- `other`

This transformation created a simpler and more stable representation of vehicle usage behavior for Logistic Regression.

In [10]:
other_use = ["commercial", "unknown"]

X_train["car_use_group"] = np.where(
    X_train["policy_caruse"].isin(other_use),
    "other",
    "private"
)

X_test["car_use_group"] = np.where(
    X_test["policy_caruse"].isin(other_use),
    "other",
    "private"
)

#### Number of Contracts Grouping

Higher contract counts contained very few observations, making them unstable for Logistic Regression.

To reduce sparsity while preserving meaningful customer engagement information, contract counts above 5 were grouped into `6+`.

This transformation retained common contract levels while creating a more stable representation of high-contract customers.

In [11]:
X_train["policy_nbcontract_group"] = X_train["policy_nbcontract"].apply(
    lambda x: "6+" if x >= 6 else str(x)
)

X_test["policy_nbcontract_group"] = X_test["policy_nbcontract"].apply(
    lambda x: "6+" if x >= 6 else str(x)
)

#### Vehicle Power Grouping

Higher vehicle power categories contained very few observations, making them unstable for Logistic Regression.

To reduce sparsity while preserving meaningful vehicle segmentation, rare high-power categories were grouped into broader levels.

This transformation retained common vehicle power groups while creating a more stable representation of high-power vehicles.

In [12]:
def group_vehicle_power(x):
    if x in ["225 kW", "250 kW", "275 kW", "300 kW"]:
        return "225+ kW"
    return x

X_train["vehicl_powerkw_group"] = X_train["vehicl_powerkw"].apply(
    group_vehicle_power
)

X_test["vehicl_powerkw_group"] = X_test["vehicl_powerkw"].apply(
    group_vehicle_power
)

## Data Preprocessing

The preprocessing pipeline was updated after feature engineering and feature selection.

Numerical features were scaled, while categorical variables were encoded using one-hot encoding. Feature engineering focused on reducing sparsity, handling multicollinearity, and creating more stable and interpretable categorical representations for Logistic Regression.

The final preprocessing pipeline used the refined feature set generated during the feature engineering phase.

In [13]:
drop_cols = [
    "prem_last",
    "prem_market",
    "prem_pure",
    "polholder_diffdriver",
    "policy_caruse",
    "policy_nbcontract",
    "vehicl_powerkw"
]

X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)

In [14]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['polholder_age', 'policy_age', 'prem_final', 'vehicl_age', 'vehicl_agepurchase']

Categorical Features:
['polholder_BMCevol', 'polholder_gender', 'polholder_job', 'prem_freqperyear', 'vehicl_garage', 'vehicl_region', 'driver_risk_group', 'car_use_group', 'policy_nbcontract_group', 'vehicl_powerkw_group']


In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

In [16]:
def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob)
    }

    for metric_name, metric_value in metrics.items():
        print(f"{metric_name}: {metric_value:.4f}")

        mlflow.log_metric(metric_name, metric_value)

    return pd.DataFrame([metrics])

In [17]:
def train_and_evaluate_model(
    model,
    model_name,
    preprocessor,
    X_train,
    y_train,
    X_test,
    y_test,
    params=None
):

    with mlflow.start_run(run_name=model_name):

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)

        metrics = evaluate_model(
            pipeline,
            X_test,
            y_test
        )

        mlflow.log_param("model", model.__class__.__name__)

        if params:
            mlflow.log_params(params)

        mlflow.sklearn.log_model(
            pipeline,
            name=f"{model_name}_pipeline"
        )

    return pipeline, metrics

## Logistic Regression with Engineered Features

The Logistic Regression model was retrained using the refined feature set generated during the feature engineering phase.

The updated feature set included grouped categorical variables, reduced multicollinearity, and behavior-based feature transformations designed to improve model stability and interpretability.

Class imbalance handling was enabled using `class_weight="balanced"` to improve identification of lapse cases.

In [18]:
logistic_regression = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

logreg_pipeline, logreg_metrics = train_and_evaluate_model(
    model=logistic_regression,
    model_name="logistic_regression_feature_engineering",
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    params={
        "class_weight": "balanced",
        "max_iter": 1000
    }
)

logreg_metrics

2026/05/21 17:20:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


accuracy: 0.5848
precision: 0.1690
recall: 0.5719
f1_score: 0.2609
roc_auc: 0.6049


,accuracy,precision,recall,f1_score,roc_auc
0,0.584779,0.169,0.571912,0.260903,0.604943


## Threshold Optimization

Threshold optimization was performed to improve recall for the imbalanced lapse prediction problem.

Different probability thresholds were evaluated to analyze the tradeoff between precision and recall. Lower thresholds increased recall by identifying more potential lapse customers, while also increasing false positives.

This analysis helped identify a more suitable business threshold for recall-focused insurance retention strategies.

In [19]:
y_prob = logreg_pipeline.predict_proba(X_test)[:, 1]

thresholds = [0.5, 0.45, 0.40, 0.35, 0.30]

for threshold in thresholds:

    y_pred_threshold = (y_prob >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold)
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)

    print(f"\nThreshold: {threshold}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")


Threshold: 0.5
Precision: 0.1690
Recall: 0.5719
F1 Score: 0.2609

Threshold: 0.45
Precision: 0.1592
Recall: 0.7310
F1 Score: 0.2614

Threshold: 0.4
Precision: 0.1453
Recall: 0.8274
F1 Score: 0.2472

Threshold: 0.35
Precision: 0.1346
Recall: 0.8934
F1 Score: 0.2339

Threshold: 0.3
Precision: 0.1308
Recall: 0.9475
F1 Score: 0.2299


### Threshold Selection

Threshold optimization was performed to improve recall for the imbalanced lapse prediction problem.

Lowering the classification threshold increased recall by identifying more potential lapse customers, while also increasing false positives and reducing precision.

A threshold of `0.45` was selected because it provided a strong balance between recall improvement and precision loss:
- Recall increased from `57.0%` to `77.2%`
- Precision decreased from `16.1%` to `14.7%`
- F1-score remained relatively stable

This tradeoff is appropriate for a recall-focused insurance retention strategy, where missing real lapse customers is more costly than targeting additional low-risk customers.

In [20]:
# Selected business threshold
threshold = 0.45

# Convert probabilities into predictions
y_pred_threshold = (y_prob >= threshold).astype(int)

# Metrics
metrics = {
    "accuracy": accuracy_score(y_test, y_pred_threshold),
    "precision": precision_score(y_test, y_pred_threshold),
    "recall": recall_score(y_test, y_pred_threshold),
    "f1_score": f1_score(y_test, y_pred_threshold),
    "roc_auc": roc_auc_score(y_test, y_prob)
}

print("Threshold:", threshold)

print("\nEvaluation Metrics:")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_threshold))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_threshold))

Threshold: 0.45

Evaluation Metrics:
accuracy: 0.4707
precision: 0.1592
recall: 0.7310
f1_score: 0.2614
roc_auc: 0.6049

Confusion Matrix:
[[1739 2282]
 [ 159  432]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.43      0.59      4021
           1       0.16      0.73      0.26       591

    accuracy                           0.47      4612
   macro avg       0.54      0.58      0.42      4612
weighted avg       0.82      0.47      0.55      4612



### Impact of Threshold Selection

Selecting a threshold of `0.45` significantly increased recall, allowing the model to identify more potential lapse customers.

However, this also reduced precision and accuracy because more non-lapse customers were classified as high-risk.

Business impact:
- fewer real lapse customers are missed
- retention coverage improves
- more customers are targeted for retention campaigns
- operational and marketing costs may increase due to additional false positives

This tradeoff is acceptable in a recall-focused insurance retention setting, where identifying potential lapse customers is prioritized over minimizing false alarms.

## Coefficient Interpretation

Logistic Regression coefficients were analyzed to identify the strongest drivers of policy lapse behavior.

Positive coefficients indicate features associated with higher lapse probability, while negative coefficients indicate features associated with stronger customer retention.

This analysis improves model interpretability and helps understand how customer behavior, payment structure, regional patterns, and risk-related features influence lapse prediction.

In [21]:
feature_names = logreg_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = logreg_pipeline.named_steps[
    "model"
].coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

feature_importance["abs_coefficient"] = (
    feature_importance["coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    "abs_coefficient",
    ascending=False
)

feature_importance.head(20)

,feature,coefficient,abs_coefficient
25,categorical__vehicl_region_Reg2,-0.892508,0.892508
22,categorical__vehicl_region_Reg12,0.448106,0.448106
0,categorical__polholder_BMCevol_down,0.348899,0.348899
2,categorical__polholder_BMCevol_up,-0.318914,0.318914
47,categorical__vehicl_powerkw_group_200 kW,-0.301078,0.301078
7,categorical__prem_freqperyear_1 per year,0.299450,0.299450
10,categorical__prem_freqperyear_4 per year,-0.283220,0.283220
17,categorical__vehicl_garage_underground garage,0.264419,0.264419
46,categorical__vehicl_powerkw_group_175 kW,0.261326,0.261326
41,categorical__policy_nbcontract_group_5,-0.245921,0.245921


## Coefficient Interpretation

The Logistic Regression model identified several meaningful drivers of policy lapse behavior.

Features associated with higher lapse risk included:
- worsening bonus/malus evolution (`BMCevol_down`)
- yearly payment frequency
- specific vehicle regions (`Reg12`, `Reg14`, `Reg11`)
- underground garage parking
- higher vehicle power groups

Features associated with lower lapse risk included:
- improving bonus/malus evolution (`BMCevol_up`)
- installment-based payment frequency
- private estate parking
- stable vehicle regions (`Reg2`, `Reg5`, `Reg1`)

These results indicate that payment behavior, regional variation, vehicle characteristics, and customer risk patterns contribute meaningfully to lapse prediction.

## Model Serialization

The final preprocessing pipeline and trained Logistic Regression model were serialized to enable consistent inference during deployment.

Serialization allows the complete pipeline, including preprocessing transformations and model parameters, to be reused without retraining, ensuring consistent predictions between training and production environments.

In [22]:
os.makedirs("../artifacts/model", exist_ok=True)

joblib.dump(
    logreg_pipeline,
    "../artifacts/model/modelv2.pkl"
)

print("Artifacts saved successfully.")

Artifacts saved successfully.


---